In [1]:
import torch
import optuna
from optuna.visualization import plot_optimization_history, plot_param_importances, plot_slice, plot_parallel_coordinate, plot_contour
from pathlib import Path
from run_with_lightning import train_and_test, train_and_test_from_yaml, load_and_test
from config import save_trial_config_from_study

In [2]:
torch.set_float32_matmul_precision('high') # will use tensor cores if available, otherwise will use float32 precision
seed = 42 # use 42 for reproducibility
data_config = {
    'batch_size' : 128, # use 128 for reproducibility
    'augment' : False} # whether to augment the data during training
# we will use augmentations later for CNNs
studies_dir = 'optuna_databases' # directory to save/load optuna studies to/from
configs_dir = 'configs' # directory to sove/load hyperparamter configurations to/from

Here we compare the optimized architectures.
First, the basic fixed MLP, `MLPBasic`:

In [ ]:
architecture_type = 'mlp_basic'
# see architectures.py for the definition of MLPBasic, or below for an outline
net_params = {} # MLPBasic is a fixed architecture, so no parameters are needed
lr = 0.01 #learning rate
weight_decay = 0.001
optimizer_params = {'lr': lr, 'weight_decay': weight_decay}
lit_module_config = {'architecture_type': architecture_type, 'net_params': net_params, 'optimizer_params': optimizer_params}
train_and_test(lit_module_config=lit_module_config, data_config=data_config, seed=seed)
# we also provide the trained model weights directly to test without re-training
# uncomment below to do that
# mlp_basic_model_path = Path('lightning_logs') / 'mlp_basic' / 'checkpoints' / 'epoch=19-step=7500.ckpt'
# load_and_test(mlp_basic_model_path, data_config=data_config, seed=seed)

We got test accuracy of 0.870. This sets the baseline for us. Let's see if we can improve on the MLP architecture and optimizer with Optuna.

running `python -m optumize --mlp` will start a big Optuna optimization study, where 100 different hyperparameter configurations will be explored efficiently using TPE (tree Parzen estimators) sampler and the median pruner.

We already run the study, the results were automatically saved in `optuna_databases/mlp.db` database. The best hyperparameter configuration is automatically saved in `mlp_best_params.yaml` we can test this directly with `train_test_from_yaml('config_path'='mlp_best_params.yaml', 'data_config'=data_config)` (remember, for hyperparameter optimization we only use the train and validation dataset, but what we are really interested in is the performance on the test dataset).

But first, let's visualize the Optuna study:

In [ ]:
studies_dir = "optuna_databases"
mlp_study_name = "mlp"
storage_name = f"sqlite:///{studies_dir}/{mlp_study_name}.db"
study_mlp = optuna.load_study(study_name=mlp_study_name, storage=storage_name)

In [ ]:
plot_optimization_history(study_mlp)

Looks like Optuna got lucky with a good guess on the very first trial and only marginally improved from there, exploring many unsuccessful configurations inbetween.

Slice plot is a great tool to see which parameters work best and which don't:

In [ ]:
plot_slice(study_mlp)

We clearly found the best learning rate. Weight decay and number of layers don't seem to matter too much, but perhaps we were too conservative with maximum number of hidden units per hidden layer. We see clear improvement the more hidden units we use up until the maximum we set at 256 in the search space.

This is typical when doing a first hyperparameter optimization study on a dataset and architecture you are not familiar with. We ran another study where we fixed the learning rate at 1e-3, weight decay at 1e-5, restricted number of hidden layers to 4 maximum, but relaxed the maximum number of hidden units up to 1024. Let's see how that went:

In [ ]:
mlp2_study_name = "mlp2"
storage_name = f"sqlite:///{studies_dir}/{mlp2_study_name}.db"
study_mlp2 = optuna.load_study(study_name=mlp2_study_name, storage=storage_name)

In [ ]:
plot_slice(study_mlp2)

While we still see improvement with increasing the number of hidden units, the improvement is very small, still just above 0.90, at a high computational cost. The differences are well within random variation from run to run.
This means we reached the limit of what MLP architecture can give us on this dataset. The best configuration was saved automatically in `mlp2_best_params.yaml`. Let's see how the best configuration does on the test dataset:

In [ ]:
mlp2_best_path = Path(configs_dir) / 'mlp2_best_params.yaml' # platform-agnostic
train_and_test_from_yaml(config_path=mlp2_best_path, data_config=data_config, seed=seed)

With hyperparameter optimization, we went from under 0.870 to 0.894 test accuracy for the MLP architecture.

Let's see what a basic fixed CNN can do with a good guess for learning rate and weight decay. CNNs are more powerful, so we will add data augmentation here: random flips and small scaling during training. See the datamodule definition in `lightning_definitions.py` for full details.

In [ ]:
architecture_type = 'cnn_basic'
# see architectures.py for the definition of CNNBasic, or below for an outline
net_params = {} # CNNBasic is a fixed architecture, so no parameters are needed
lr = 0.01 # learning rate
weight_decay = 0.001
optimizer_params = {'lr': lr, 'weight_decay': weight_decay}
lit_module_config = {'architecture_type': architecture_type, 'net_params': net_params, 'optimizer_params': optimizer_params}
data_config = {
    'batch_size' : 128, # use 128 for reproducibility
    'augment' : True} # turn on augmentation for CNNs
train_and_test(lit_module_config=lit_module_config, data_config=data_config, seed=seed)
# we also provide the trained model weights directly to test without re-training
# uncomment below to do that
# cnn_basic_model_path = Path('lightning_logs') / 'cnn_basic' / 'checkpoints' / 'epoch=27-step=10500.ckpt'
# load_and_test(cnn_basic_model_path, data_config=data_config, seed=seed)

This is not fully reproducible on CUDA, but should give about 0.88 test accuracy. So, better than the basic MLP architecture we started with, but worse than the optimized MLP.

Hopefully we can improve on this with hyperparameter optimization.

running `python -m optumize --cnn` will again start a big Optuna optimization study for the CNN architecture.

We already run the study, the results were automatically saved in `optuna_databases/cnn2.db` database. The best hyperparameter configuration is automatically saved in `cnn2_best_params.yaml`. Let's test the best configuration.

In [ ]:
cnn2_best_path = Path(configs_dir) / 'cnn2_best_params.yaml'
train_and_test_from_yaml(config_path=cnn2_best_path, data_config=data_config, seed=seed)

We got over 0.945 test accuracy, but the model is huge! 17.3M parameters, 69MB estimated model size. Let's take a look at the Optuna study and see if we can get something cheaper.

In [ ]:
cnn2_study_name = "cnn2"
storage_name = f"sqlite:///{studies_dir}/{cnn2_study_name}.db"
study_cnn2 = optuna.load_study(study_name=cnn2_study_name, storage=storage_name)
df = study_cnn2.trials_dataframe()
df_sorted = df.sort_values("value")
df_sorted.tail(10)

Trial 45 has nearly the same validation performance, but uses half the out_channels. We can save this configuration to test it:

In [4]:
save_trial_config_from_study(study_cnn2, "cnn2", 45)

In [ ]:
data_config = {
    'batch_size' : 128, # use 128 for reproducibility
    'augment' : True} # turn on augmentation for CNNs
train_and_test_from_yaml(config_path=f'{configs_dir}/cnn2_trial45_config.yaml', data_config=data_config, seed=seed)

cnn2_trial45 is 4 times smaller than the best but has a comparable test accuracy at 0.94

We launched another study, this time fixing out_channels at 64 and n_intermediate to 1 to limit the model size. Let's test the best configuration from that study. 

In [ ]:
data_config = {
    'batch_size' : 128, # use 128 for reproducibility
    'augment' : True} # turn on augmentation for CNNs
cnn_64_best_path = Path(configs_dir) / 'cnn_64_best_params.yaml'
train_and_test_from_yaml(config_path=cnn_64_best_path, data_config=data_config, seed=seed)
# we also provide the trained model weights directly to test without re-training
# uncomment below to do that
# cnn_64_model_path = Path('lightning_logs') / 'cnn_64_best_params' / 'checkpoints' / 'epoch=52-step=19875.ckpt'
# load_and_test(cnn_64_model_path, data_config=data_config, seed=seed)

408K params, test accuracy of about 0.936 (not fully reproducible on CUDA). A good compromise.

All learning curves and test results can be visualized by running `tensorboard --logdir lightning_logs`

![Validation and test accuracies](val_test_accuracy.png)